# HANAS Phase 3 Operational Evaluation

Phase 3 evaluates the Agentic AI controller during hydroponic lettuce cultivation. The analysis covers pH and EC regulation, dosing response, recorded decisions, and crop outcomes.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

def find_research_analysis():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        direct = candidate / 'raw_data' / 'phase3' / 'Phase 3 - Production Result.xlsx'
        nested = candidate / 'research_analysis' / 'raw_data' / 'phase3' / 'Phase 3 - Production Result.xlsx'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'research_analysis'
    raise FileNotFoundError('Could not locate research_analysis.')

ROOT = find_research_analysis()
WORKBOOK = ROOT / 'raw_data' / 'phase3' / 'Phase 3 - Production Result.xlsx'
CROP_WORKBOOK = ROOT / 'raw_data' / 'phase3' / 'Phase 3 - Lettuce Measurement.xlsx'
SCRIPT = ROOT / 'scripts' / 'analyze_phase3_standard_metrics.py'
OUTPUT = ROOT / 'reports' / 'phase3_figures' / 'phase3_integrity_metrics.json'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print('Production workbook:', WORKBOOK)


## Reproducible operational metrics

Scheduled decisions that reuse a sensor reading are not counted as new measurements. Maintenance-mode rows remain in the control audit but are excluded from nutrient-compliance estimates. Long data gaps are excluded and reported separately, so compliance covers only the evaluated periods.


In [ ]:
completed = subprocess.run(
    [sys.executable, str(SCRIPT), str(WORKBOOK), str(OUTPUT)],
    check=True, capture_output=True, text=True,
)
metrics = json.loads(completed.stdout)
display(pd.DataFrame([
    ('Source SHA-256', metrics['source_workbook_sha256']),
    ('System-log rows', metrics['data_integrity']['system_log_rows']),
    ('Control-cycle rows', metrics['data_integrity']['control_cycle_rows']),
    ('Duplicate/orphan/parse errors', sum(metrics['data_integrity'][key] for key in ['duplicate_system_log_ids','duplicate_control_cycle_ids','orphan_control_cycle_links','decision_metadata_parse_errors'])),
    ('Operational observations', metrics['operational_observations']),
    ('Evaluated hours', metrics['evaluated_hours']),
    ('Evaluated coverage (%)', metrics['evaluated_time_coverage_pct']),
    ('Joint evaluated-time compliance (%)', metrics['joint_time_in_range_pct']),
    ('Full-span lower bound (%)', metrics['full_span_joint_lower_bound_pct']),
    ('Full-span upper bound (%)', metrics['full_span_joint_upper_bound_pct']),
    ('Excluded gaps', metrics['excluded_inter_snapshot_gaps']),
    ('Gaps starting out of range', metrics['gaps_starting_out_of_range']),
    ('Duration-evaluable excursions', metrics['joint_excursions_with_evaluable_duration']['count']),
    ('Mean nutrient-solution temperature (deg C)', metrics['mean_reservoir_temperature_c']),
    ('Minimum reservoir volume (L)', metrics['observed_reservoir_volume_liters'][0]),
    ('Maximum reservoir volume (L)', metrics['observed_reservoir_volume_liters'][1]),
], columns=['Metric', 'Value']))


## Command-response uncertainty


In [ ]:
response = metrics['system_command_response']
rows = []
for label, key in [
    ('First post-mixing attainment', 'first_post_mixing_attainment'),
    ('Attainment before repeat', 'attainment_before_repeat'),
    ('Repeat correction required', 'repeat_correction_required'),
    ('First ten post-mixing observations in range', 'sustained_ten_observations'),
]:
    value = response[key]
    rows.append({
        'Metric': label, 'Count': value['count'], 'N': value['total'],
        'Percent': value['pct'], 'Wilson 95% lower': value['lower_95_pct'],
        'Wilson 95% upper': value['upper_95_pct'],
    })
display(pd.DataFrame(rows).round(2))


## Crop outcomes (descriptive; plants share one reservoir)


In [ ]:
harvest = pd.read_excel(CROP_WORKBOOK, sheet_name='day_35', engine='openpyxl')
weights = pd.to_numeric(harvest['Shoot Fresh Weight (g)'], errors='coerce')
surviving = harvest['Status'].str.lower().eq('alive').sum()
n_plants = len(harvest)
crop_summary = pd.DataFrame({
    'Metric': ['Plants', 'Surviving', 'Observed survival (%)', 'Mean weight (g)', 'SD (g)', 'CV (%)', 'Total weight (g)'],
    'Value': [n_plants, surviving, surviving/n_plants*100, weights.mean(), weights.std(ddof=1), weights.std(ddof=1)/weights.mean()*100, weights.sum()],
})
display(crop_summary.round(2))
display(harvest.groupby('Channel')['Shoot Fresh Weight (g)'].agg(['count','mean','std','min','max','sum']).round(2))


## Limits of interpretation

These results describe one cultivation cycle without a comparison controller. They do not show that the controller outperforms other approaches or improves crop production. They also do not establish how accurately the safety gate detects unsafe conditions. Recorded dose commands do not independently confirm the volume delivered or show that dosing alone caused later changes in pH or EC.

Stability filtering can exclude readings taken while conditions are changing. Report compliance for the evaluated periods alongside data coverage and the bounds for the full monitoring period. Compliance during evaluated periods should not be assumed to represent the whole cultivation cycle.
